## 🎯 Learning Objectives
* Understand the fundamental components of a Reinforcement Learning (RL) training loop.
* Implement a Q-learning agent to interact with a discrete Gym environment.
* Apply an epsilon-greedy policy for effective exploration and exploitation.
* Evaluate the performance of a trained RL agent using metrics like average reward and success rate.


## Exercise: Train an Agent on a Simple Gym Environment

**Task:** Your goal is to implement a Q-learning agent to solve a classic discrete Reinforcement Learning environment from the `gymnasium` library. We will use the `FrozenLake-v1` environment (non-slippery version) for this exercise, as it provides a clear, grid-world setting to focus on the core Q-learning algorithm.

**Environment Description (`FrozenLake-v1`):**
*   **Observation Space:** A discrete integer representing the agent's current position on a 4x4 grid (0 to 15).
*   **Action Space:** Discrete actions: 0 (LEFT), 1 (DOWN), 2 (RIGHT), 3 (UP).
*   **Rewards:**
    *   +1 for reaching the goal (G).
    *   0 for reaching a frozen state (F).
    *   0 for falling into a hole (H).
*   **Goal:** Navigate from the start (S) to the goal (G) without falling into holes.

**Requirements:**
1.  **Environment Setup:** Initialize the `FrozenLake-v1` environment with `is_slippery=False` to make it deterministic.
2.  **Q-Table Initialization:** Create a Q-table (NumPy array) initialized with zeros, with dimensions `(observation_space_size, action_space_size)`.
3.  **Hyperparameters:** Define appropriate hyperparameters for Q-learning, including:
    *   `learning_rate` (alpha)
    *   `discount_factor` (gamma)
    *   `epsilon` (for exploration)
    *   `epsilon_decay_rate`
    *   `min_epsilon`
    *   `num_episodes` for training
4.  **Epsilon-Greedy Policy:** Implement a function or logic to select actions using an epsilon-greedy strategy. This means with probability `epsilon`, a random action is chosen; otherwise, the action with the highest Q-value for the current state is chosen.
5.  **Q-Value Update:** Implement the Q-learning update rule:
    `Q(s, a) = Q(s, a) + learning_rate * [reward + discount_factor * max(Q(s', a')) - Q(s, a)]`
6.  **Training Loop:** Iterate through a specified number of episodes. Within each episode, simulate steps until the episode terminates (agent reaches goal, falls in hole, or max steps reached).
7.  **Epsilon Decay:** Decrease `epsilon` over time to reduce exploration as the agent learns.
8.  **Evaluation:** After training, evaluate the agent's performance over a set number of test episodes (e.g., 100 episodes) without exploration (i.e., `epsilon=0`). Calculate and report the average reward and success rate.

**Evaluation Criteria:**
*   **Correctness:** The Q-learning algorithm is correctly implemented, and the Q-table is updated as expected.
*   **Performance:** The agent should achieve a high success rate (e.g., > 70%) on the non-slippery `FrozenLake-v1` environment after training.
*   **Code Quality:** Code is well-structured, readable, and includes comments where necessary.


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

# 1. Environment Setup
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode=None) # No rendering during training

# Get observation and action space sizes
observation_space_size = env.observation_space.n
action_space_size = env.action_space.n

print(f"Observation Space Size: {observation_space_size}")
print(f"Action Space Size: {action_space_size}")

# 2. Q-Table Initialization
# Initialize Q-table with zeros. This table will store the maximum expected future rewards
# for taking an action in a given state.
q_table = np.zeros((observation_space_size, action_space_size))

# 3. Hyperparameters
# Learning rate (alpha): How much new information overrides old information.
learning_rate = 0.1

# Discount factor (gamma): How much future rewards are valued compared to immediate rewards.
discount_factor = 0.99

# Exploration-exploitation trade-off parameters
epsilon = 1.0          # Initial exploration rate
epsilon_decay_rate = 0.0001 # Rate at which epsilon decays per episode
min_epsilon = 0.01     # Minimum exploration rate

# Training parameters
num_episodes = 20000   # Total number of episodes for training
max_steps_per_episode = 100 # Maximum steps an agent can take in an episode

print("Setup complete. Q-table initialized and hyperparameters defined.")
print("Q-table shape:", q_table.shape)


## Your Turn! Implement the Q-learning Algorithm

Now it's time to put your knowledge into practice. Below, you'll find the setup code we've prepared. Your task is to complete the training loop for the Q-learning agent.

**Instructions:**
1.  **Implement the Epsilon-Greedy Action Selection:** Within each step of an episode, decide whether to explore (take a random action) or exploit (take the best known action from the Q-table) based on the current `epsilon` value.
2.  **Execute a Step:** Use `env.step(action)` to interact with the environment, receiving the `new_state`, `reward`, `terminated`, `truncated`, and `info`.
3.  **Update Q-Table:** Apply the Q-learning update rule using the `learning_rate`, `discount_factor`, current `state`, `action`, `reward`, and `new_state`.
4.  **Handle Episode Termination:** Break the inner loop if the episode `terminated` or `truncated`.
5.  **Decay Epsilon:** After each episode, update `epsilon` using the `epsilon_decay_rate` and ensure it doesn't fall below `min_epsilon`.
6.  **Store Rewards:** Keep track of rewards per episode to monitor learning progress.
7.  **Evaluate:** After the training loop, implement a separate evaluation phase where the agent acts purely greedily (no exploration) for a set number of test episodes. Calculate and print the average reward and success rate.

Feel free to add print statements or plot the rewards to visualize the learning process.


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

# --- Setup (as provided in the previous cell) ---
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode=None)
observation_space_size = env.observation_space.n
action_space_size = env.action_space.n
q_table = np.zeros((observation_space_size, action_space_size))

learning_rate = 0.1
discount_factor = 0.99
epsilon = 1.0
epsilon_decay_rate = 0.0001
min_epsilon = 0.01
num_episodes = 20000
max_steps_per_episode = 100
# --------------------------------------------------

rewards_per_episode = []

# --- Training Loop ---
print("\nStarting Q-learning training...")
for episode in range(num_episodes):
    # Reset the environment for a new episode
    state, info = env.reset()
    terminated = False  # True when agent reaches goal or falls in hole
    truncated = False   # True when episode exceeds max_steps_per_episode
    current_episode_reward = 0

    for step in range(max_steps_per_episode):
        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon: # Explore: choose a random action
            action = env.action_space.sample()
        else: # Exploit: choose the action with the highest Q-value for the current state
            action = np.argmax(q_table[state, :])

        # Take the chosen action and observe the new state and reward
        new_state, reward, terminated, truncated, info = env.step(action)

        # Q-learning update rule
        # Q(s, a) = Q(s, a) + learning_rate * [reward + discount_factor * max(Q(s', a')) - Q(s, a)]
        q_table[state, action] = q_table[state, action] + learning_rate * \
                                 (reward + discount_factor * np.max(q_table[new_state, :]) - q_table[state, action])

        # Update current state and accumulate reward
        state = new_state
        current_episode_reward += reward

        # Check if the episode has ended
        if terminated or truncated:
            break

    # Decay epsilon after each episode to reduce exploration over time
    epsilon = max(min_epsilon, epsilon - epsilon_decay_rate)

    # Store the total reward for this episode
    rewards_per_episode.append(current_episode_reward)

    # Optional: Print progress
    if (episode + 1) % 1000 == 0:
        avg_reward = np.mean(rewards_per_episode[-1000:])
        print(f"Episode {episode + 1}/{num_episodes} | Epsilon: {epsilon:.4f} | Avg Reward (last 1000): {avg_reward:.2f}")

env.close()
print("Training complete!")

# --- Evaluation Phase ---
print("\nStarting evaluation...")
num_test_episodes = 100
successful_episodes = 0
total_test_rewards = []

env_eval = gym.make("FrozenLake-v1", is_slippery=False, render_mode=None) # Re-create env for evaluation

for episode in range(num_test_episodes):
    state, info = env_eval.reset()
    terminated = False
    truncated = False
    episode_reward = 0

    for step in range(max_steps_per_episode):
        # For evaluation, always choose the greedy action (no exploration)
        action = np.argmax(q_table[state, :])
        new_state, reward, terminated, truncated, info = env_eval.step(action)

        episode_reward += reward
        state = new_state

        if terminated or truncated:
            if reward == 1: # Assuming reward of 1 means success in FrozenLake
                successful_episodes += 1
            break
    total_test_rewards.append(episode_reward)

env_eval.close()

# Calculate and print evaluation metrics
average_test_reward = np.mean(total_test_rewards)
success_rate = (successful_episodes / num_test_episodes) * 100

print(f"\n--- Evaluation Results ({num_test_episodes} episodes) ---")
print(f"Average Reward: {average_test_reward:.2f}")
print(f"Success Rate: {success_rate:.2f}%")

# --- Visualization of Training Progress ---
plt.figure(figsize=(12, 6))
plt.plot(rewards_per_episode)
plt.title('Reward per Episode during Training')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.grid(True)
plt.show()

# Optional: Plot average rewards over windows to smooth out noise
window_size = 100
avg_rewards_smoothed = np.convolve(rewards_per_episode, np.ones(window_size)/window_size, mode='valid')
plt.figure(figsize=(12, 6))
plt.plot(avg_rewards_smoothed)
plt.title(f'Smoothed Average Reward (Window Size: {window_size})')
plt.xlabel('Episode (Smoothed)')
plt.ylabel('Average Reward')
plt.grid(True)
plt.show()

print("\nFinal Q-table (first 5 states):\n", q_table[:5])
